# 01 · Fundamentos de Scala desde cero

> Este archivo es la base de todos los demás. Si nunca has escrito Scala, léelo entero
> una vez. Después úsalo como diccionario: cada vez que en otro cursillo aparezca algo
> raro, está explicado aquí.

Se asume que sabes programar en algún lenguaje (Java, Python, C…), pero **no** que sepas
nada de Scala.

---

## 0. Cómo se trabaja en la asignatura

- Los ejercicios y exámenes son **notebooks de Jupyter** (`.ipynb`) con el kernel de Scala
  *almond*. Cada celda se ejecuta por separado y lo que defines en una celda queda
  disponible para las siguientes.
- Se usa **Scala 3**. Los exámenes antiguos (19-20 a 23-24) están escritos con la sintaxis
  de **Scala 2** (llaves `{ }`). Las dos sintaxis funcionan; aquí verás ambas.
- Al principio de cada examen hay un **preámbulo** con definiciones que te dan hechas
  (`Tree`, `Isomorphic`, `Not`, `ExcludedMiddle`…). Léelo siempre: te dice qué puedes usar.

---

## 1. Valores: `val`

In [ ]:
val x: Int = 3
val nombre = "Ana"      // el tipo (String) se deduce solo

- `val` crea un **valor inmutable**: no se puede reasignar. (Existe `var`, que sí se puede
  reasignar, pero en programación funcional **no se usa**; solo aparece en los enunciados
  como "versión imperativa" que te piden traducir.)
- `x: Int` se lee "x es de tipo Int". En Scala **el tipo va detrás** del nombre, separado
  por dos puntos.
- Si no pones el tipo, Scala lo deduce (*inferencia de tipos*).

### Tipos básicos

| Tipo | Ejemplos | Comentario |
|---|---|---|
| `Int` | `0`, `-3`, `42` | enteros |
| `Double` | `3.14`, `2.0` | decimales |
| `Boolean` | `true`, `false` | |
| `Char` | `'a'`, `'X'` | comillas **simples** |
| `String` | `"hola"` | comillas **dobles** |
| `Unit` | `()` | tiene **un único valor**, `()`. Es como `void`, pero es un tipo de verdad |
| `Nothing` | (ninguno) | **no tiene ningún valor**. Clave en Curry-Howard (archivo 02) |

---

## 2. Funciones: `def`

In [ ]:
def suma(a: Int, b: Int): Int =
    a + b

Se lee: "defino `suma`, que recibe `a` de tipo `Int` y `b` de tipo `Int`, y devuelve un
`Int`; su resultado es `a + b`".

Cosas importantes que sorprenden al venir de Java:

1. **No hay `return`.** El resultado de una función es **la última expresión** de su cuerpo.
2. El cuerpo va tras el `=`. En Scala 3 basta con **indentar** (como en Python). En Scala 2
   se usan llaves:

   ```scala
   // Scala 2 (exámenes antiguos)
   def suma(a: Int, b: Int): Int = {
       a + b
   }
   ```

3. Dentro de una función puedes definir valores y otras funciones auxiliares:

   ```scala
   def f(n: Int): Int =
       val doble = n * 2          // valor local
       def mas1(k: Int): Int = k + 1   // función local (auxiliar)
       mas1(doble)                // última expresión = resultado
   ```

### `if` es una expresión (devuelve un valor)

In [ ]:
def abs(n: Int): Int =
    if n < 0 then -n else n          // Scala 3
// Scala 2:  if (n < 0) -n else n

No existe el operador `?:` de Java porque `if/else` ya hace ese papel.

### Varias listas de parámetros (currificación)

In [ ]:
def suma(a: Int)(b: Int): Int = a + b

suma(1)(2)       // 3

Es lo mismo que una función de dos argumentos, pero se llaman por separado. Se usa
muchísimo en la asignatura (por ejemplo `foldRight(valorInicial)(funcion)`), porque:

- permite fijar el primer argumento y obtener una función: `suma(1)` "espera" el `b`;
- ayuda a Scala a deducir los tipos: lo que aprende en la primera lista lo usa en la segunda.

---

## 3. Genéricos: `[A]`, `[A, B]`

In [ ]:
def identidad[A](a: A): A = a

identidad(3)        // A = Int
identidad("hola")   // A = String

`[A]` declara un **parámetro de tipo**: una variable que representa "un tipo cualquiera".
Es como `<T>` en Java. En los exámenes casi todas las funciones son genéricas:

In [ ]:
def proof[P, Q]: ...                     // P y Q son "proposiciones" (tipos cualesquiera)
def swap[A, B](l: List[Either[A, B]]): Either[List[A], List[B]]

Al llamar a la función normalmente **no** hace falta decir los tipos; Scala los deduce.
Si alguna vez hay que decirlos, se ponen entre corchetes: `lem[P]`, `foldRight[Int](0)(...)`.

---

## 4. Funciones como valores (lambdas)

En Scala una función es un valor más: se guarda en variables, se pasa como argumento y se
devuelve como resultado.

### El tipo de una función: `A => B`

`Int => Boolean` es "el tipo de las funciones que reciben un `Int` y devuelven un `Boolean`".

### Escribir una función anónima (lambda)

In [ ]:
val esPar: Int => Boolean = (n: Int) => n % 2 == 0
val esPar2: Int => Boolean = n => n % 2 == 0     // el tipo de n se deduce del tipo declarado
esPar(4)   // true

La flecha `=>` separa los **parámetros** (izquierda) del **cuerpo** (derecha).

Con dos parámetros:

In [ ]:
val suma: (Int, Int) => Int = (a, b) => a + b

### La abreviatura `_` (placeholder)

In [ ]:
val mas1: Int => Int = _ + 1          // equivale a  x => x + 1
val suma: (Int, Int) => Int = _ + _   // equivale a  (a, b) => a + b

**Cada `_` es un parámetro distinto**, en orden. `_ + _` NO es `x => x + x`, es
`(a, b) => a + b`. Si necesitas usar el mismo parámetro dos veces, escribe la lambda
completa (`x => x + x`).

### Funciones que devuelven funciones

In [ ]:
def sumador(n: Int): Int => Int =
    x => x + n

val mas5 = sumador(5)
mas5(10)     // 15

El tipo `A => B => C` significa `A => (B => C)`: **la flecha asocia a la derecha**. Es una
función que recibe una `A` y devuelve otra función que recibe una `B` y devuelve una `C`.
Por eso se escribe `f => g => h => ...` y se llama `f(a)(b)`.

### Función de dos parámetros vs. función que recibe una tupla

Ojo con esta sutileza (aparece en Curry-Howard):

- `(P, Q) => R` es una función de **dos** parámetros.
- `((P, Q)) => R` es una función de **un** parámetro que es una **tupla** (fíjate en el
  doble paréntesis).

En Scala 3 puedes escribir la lambda de la segunda con dos nombres y Scala "desempaqueta"
la tupla solo (*parameter untupling*):

In [ ]:
val f: ((Int, Int)) => Int = (a, b) => a + b   // válido en Scala 3

---

## 5. Tuplas (tipos producto)

In [ ]:
val t: (Int, String) = (1, "uno")
t._1       // 1      (primer componente)
t._2       // "uno"  (segundo componente)

val (n, s) = t      // "desestructurar": n = 1, s = "uno"

val triple: (Int, Char, Boolean) = (3, 'x', true)

- `(A, B)` es el tipo de los pares. Es el **"y"** lógico (∧) en Curry-Howard.
- `._1`, `._2`, `._3`… acceden a los componentes.

---

## 6. Pattern matching: `match` y `case`

Es la herramienta más importante de la asignatura. Es como un `switch` muy potente: mira la
**forma** de un valor y, a la vez, le saca las piezas.

In [ ]:
def describe(n: Int): String =
    n match
        case 0 => "cero"
        case 1 => "uno"
        case _ => "otro"     // _ = "cualquier otra cosa"

Reglas:

- Los casos se prueban **en orden, de arriba abajo**; se ejecuta el primero que encaje.
  Pon los casos particulares antes que los generales.
- `_` encaja con cualquier cosa y no le da nombre.
- Un nombre en minúscula (`x`, `head`, `a`) encaja con cualquier cosa **y le da ese nombre**.
- Puedes añadir una condición con `if` (se llama *guarda*):

  ```scala
  n match
      case x if x < 0 => "negativo"
      case _ => "no negativo"
  ```

- Si quieres comparar con el valor de una variable que ya existe (y no crear una nueva),
  escríbela entre comillas invertidas:

  ```scala
  def esIgual(a: Int, n: Int): Boolean =
      n match
          case `a` => true      // ¿n es igual a a?
          case _   => false
  ```

- `nombre@patron` da nombre al valor completo mientras lo descompones:
  `case todo@(h :: t) => ...` (todo es la lista entera, h la cabeza, t el resto).

Sintaxis Scala 2: igual, pero con llaves: `n match { case 0 => ...; case _ => ... }`.

### Desestructurar tuplas con `match`

```scala
(a, b) match
    case (0, 0) => "los dos cero"
    case (0, _) => "el primero cero"
    case (x, y) => s"$x y $y"
```

Hacer `match` sobre una tupla que construyes en el momento (`(head, f(tail)) match ...`) es
un truco que se usa constantemente para tratar varias cosas a la vez.

### Funciones hechas solo de `case`

Una función cuyo cuerpo es solo un `match` sobre su argumento se puede escribir directamente
con los `case`:

In [ ]:
// Scala 3
val sym: Char => Int =
    case 'I' => 1
    case 'V' => 5
    case 'X' => 10

// Scala 2 (y también válido en Scala 3)
val sym2: Char => Int = {
    case 'I' => 1
    case 'V' => 5
}

Esto se ve muchísimo como argumento de `foldRight`, `foldLeft`, `map`…:

```scala
lista.foldRight(0){ case (head, tailSol) => head + tailSol }   // Scala 2
lista.foldRight(0):                                             // Scala 3
    case (head, tailSol) => head + tailSol
```

(En Scala 3, `f(x):` seguido de un bloque indentado significa "pasa ese bloque como último
argumento". Es lo mismo que poner llaves.)

---

## 7. `Option`: un valor que puede no estar

In [ ]:
val a: Option[Int] = Some(3)    // hay un valor: 3
val b: Option[Int] = None       // no hay valor

`Option[A]` tiene dos formas: `Some(a)` o `None`. Sustituye a `null`.

In [ ]:
def dividir(a: Int, b: Int): Option[Int] =
    if b == 0 then None else Some(a / b)

dividir(10, 2) match
    case Some(r) => s"resultado $r"
    case None    => "no se puede"

Útiles: `opt.isDefined` (¿es `Some`?), `opt.map(f)` (aplica `f` dentro si es `Some`),
`"12".toIntOption` (convierte texto a `Option[Int]`), `Option.empty[Int]` (un `None` con su
tipo dicho). Cardinalidad (archivo 03): `Option[A]` = 1 + A.

---

## 8. `Either`: una cosa u otra

In [ ]:
val e1: Either[String, Int] = Left("error")
val e2: Either[String, Int] = Right(42)

`Either[A, B]` contiene **o bien** una `A` (envuelta en `Left`) **o bien** una `B` (envuelta
en `Right`). Es el **"o"** lógico (∨) en Curry-Howard. Se usa también para errores: por
convenio, `Left` = error, `Right` = éxito.

```scala
e2 match
    case Left(msg) => s"falló: $msg"
    case Right(n)  => s"vale $n"
```

Útiles: `e.isLeft`, `e.isRight`, `e.map(f)` (aplica `f` al `Right`).

Anidar `Either` sirve para "tres o más opciones":
`Either[A, Either[B, C]]` tiene tres formas: `Left(a)`, `Right(Left(b))`, `Right(Right(c))`.

---

## 9. Listas: `List`

In [ ]:
val l: List[Int] = List(1, 2, 3)
val vacia: List[Int] = Nil          // también List() o List[Int]()
val l2 = 0 :: l                     // List(0, 1, 2, 3)

Una lista es **inmutable** y está definida de forma recursiva. Una lista es:

- o la lista vacía, `Nil`,
- o una **cabeza** (`head`, el primer elemento) pegada con `::` a un **resto** (`tail`, otra lista).

```
List(1, 2, 3)  ==  1 :: (2 :: (3 :: Nil))
```

Por eso casi todas las funciones sobre listas tienen esta forma:

In [ ]:
def suma(l: List[Int]): Int =
    l match
        case Nil => 0                         // lista vacía
        case head :: tail => head + suma(tail) // cabeza + (suma del resto)

Patrones útiles:

| Patrón | Encaja con |
|---|---|
| `Nil` | la lista vacía |
| `head :: tail` | una lista con al menos 1 elemento |
| `x :: Nil` | una lista con exactamente 1 elemento |
| `a :: b :: tail` | una lista con al menos 2 elementos |
| `List(a, b)` | una lista con exactamente 2 elementos |
| `Left(a) :: tail` | lista no vacía cuya cabeza es un `Left` |

### Coste: añadir delante es barato, detrás es caro

- `x :: l` (añadir delante) es instantáneo.
- `l :+ x` o `l.appended(x)` (añadir al final) y `l1 ++ l2` (concatenar) tienen que
  recorrer la lista.

Por eso, cuando se construye una lista recorriendo de izquierda a derecha (recursión final,
`foldLeft`), se añade **delante** y al final se da la vuelta con `.reverse` (archivo 05).

### Métodos de lista que salen en los exámenes

| Método | Qué hace | Ejemplo |
|---|---|---|
| `length` / `size` | número de elementos | `List(1,2).length == 2` |
| `isEmpty` | ¿está vacía? | |
| `reverse` | da la vuelta | `List(1,2,3).reverse == List(3,2,1)` |
| `l1 ++ l2` | concatena | `List(1) ++ List(2) == List(1,2)` |
| `l :+ x` / `l.appended(x)` | añade al final | `List(1) :+ 2 == List(1,2)` |
| `contains(x)` | ¿contiene x? | |
| `take(n)` / `drop(n)` | primeros n / quita los primeros n | `List(1,2,3).take(2) == List(1,2)` |
| `splitAt(n)` | parte en dos en la posición n | `List(1,2,3).splitAt(1) == (List(1), List(2,3))` |
| `zipWithIndex` | empareja cada elemento con su posición | `List('a','b').zipWithIndex == List(('a',0),('b',1))` |
| `sum` | suma (lista numérica) | |
| `headOption` | `Some(cabeza)` o `None` | |
| `sliding(n)` | ventanas de tamaño n (devuelve iterador: usa `.toList`) | `List(1,2,3).sliding(2).toList == List(List(1,2), List(2,3))` |
| `flatten` | aplana una lista de listas | `List(List(1),List(2,3)).flatten == List(1,2,3)` |

`map`, `filter`, `flatMap`, `foldLeft`, `foldRight`, `exists`, `forall`… tienen sus propios
archivos (04, 05 y 06).

---

## 10. Definir tipos de datos propios

### `case class`: un registro (tipo producto)

In [ ]:
case class Punto(x: Int, y: Int)

val p = Punto(1, 2)     // no hace falta "new"
p.x                     // 1
p match
    case Punto(a, b) => a + b

### `enum` (Scala 3): un tipo con varias formas (tipo suma)

Así se define el árbol binario de los exámenes recientes:

In [ ]:
enum Tree[A]:
    case Empty()
    case Node(left: Tree[A], root: A, right: Tree[A])

import Tree._        // para poder escribir Empty() y Node(...) sin "Tree." delante

Un `Tree[A]` es **o** un árbol vacío `Empty()` **o** un nodo con hijo izquierdo, raíz e
hijo derecho. Es recursivo: los hijos son a su vez árboles.

### `sealed abstract class` + `case class` (Scala 2)

Lo mismo en la sintaxis antigua (exámenes 19-20 a 23-24):

In [ ]:
sealed abstract class Tree[A]
case class Empty[A]() extends Tree[A]
case class Node[A](left: Tree[A], root: A, right: Tree[A]) extends Tree[A]

`sealed` significa "no hay más formas que estas": así Scala te avisa si en un `match` te
olvidas de un caso. `sealed trait` hace lo mismo.

### `object`: un contenedor de funciones

In [ ]:
object Tree:
    def leaf[A](a: A): Tree[A] = Node(Empty(), a, Empty())

Agrupa funciones auxiliares. Con `import Tree._` las usas directamente: `leaf(3)`.

### `trait`: una interfaz

In [ ]:
trait Isomorphic[A, B]:
    def from(a: A): B          // sin cuerpo: hay que implementarla
    def to(b: B): A

Para implementarla: `class Iso extends Isomorphic[Int, Int]: def from(...) = ...`.
Para crear un objeto anónimo que la implemente: `new Isomorphic[Int, Int] { def from(...) = ...; def to(...) = ... }`.

### `type`: un alias (otro nombre para un tipo)

In [ ]:
type Not[P] = P => Nothing
type Nel[A] = (A, List[A])

No crea nada nuevo; `Not[P]` es solo una forma corta de escribir `P => Nothing`.

---

## 11. Otras piezas que aparecen

- **`???`**: "todavía no lo he escrito". Compila con cualquier tipo, pero falla si se
  ejecuta. Úsalo para dejar huecos mientras construyes una solución (el profesor lo usa
  en las plantillas).
- **Anotar el tipo de una expresión**: `(expr: Tipo)`. Por ejemplo `Nil: List[Int]` dice
  "este Nil es una lista de Int". Lo necesitarás en los valores iniciales de los folds.
- **Cadenas**: `"a" + "b"`, interpolación `s"vale $x"`, `"hola".toList == List('h','o','l','a')`,
  `s.length`, `"12".toIntOption`.
- **Rangos**: `1 to 5` son los números 1,2,3,4,5; `(1 to 5).toList` lo convierte en lista.
- **Igualdad**: `==` compara contenido (también listas, tuplas y case classes). `!=` es distinto.
- **Booleanos**: `&&`, `||`, `!`.
- **Resto y división entera**: `7 % 2 == 1`, `7 / 2 == 3`.
- **Parámetros con nombre**: `repeat(from = 0)(...)`. Puedes nombrar el argumento al llamar.
- **`identity`**: la función `x => x`.

---

## 12. Los tests (scalatest) y `run`

Los exámenes vienen con tests ya escritos. Tienen esta forma:

In [ ]:
class TestSwap(swap: List[Either[Int, Char]] => Either[List[Int], List[Char]])
extends AnyFlatSpec with should.Matchers:

    "swap" should "work" in:
        swap(List()) shouldBe Right(List())
        swap(List(Left(1), Left(2))) shouldBe Left(List(1, 2))

Cómo leerlo:

- `TestSwap` es una clase que **recibe tu función como parámetro**. Fíjate en el tipo que
  pide: es la signatura de lo que tienes que implementar (con tipos concretos, `Int` y `Char`).
- Cada `x shouldBe y` comprueba que `x == y`. **Los tests son la mejor especificación**:
  si el enunciado no queda claro, mira los ejemplos.
- Para ejecutarlos: `run(TestSwap(swap))`. Aunque tu `swap` sea genérico (`[A, B]`), Scala
  lo adapta solo a `Int` y `Char`.
- En exámenes antiguos verás `extends FlatSpec with Matchers{ ... }` y `run(new TestSwap(swap))`:
  es lo mismo con la sintaxis vieja.

---

## 13. Errores típicos de principiante

| Error | Causa probable | Arreglo |
|---|---|---|
| `type mismatch; found: List[Nothing] required: List[Int]` | valor inicial `Nil` sin tipo en un fold | escribe `Nil: List[Int]` o `List[Int]()` |
| `match may not be exhaustive` (aviso) | te falta un caso | añade el caso que falta (por ejemplo `Nil`) |
| `missing parameter type` | Scala no sabe el tipo de un parámetro de una lambda | anótalo: `(x: Int) => ...` |
| La función devuelve `Unit` | pusiste algo después de la expresión que querías devolver, o usaste `=` mal | la última línea debe ser el resultado |
| Resultado al revés | construiste una lista con `::` recorriendo de izquierda a derecha | añade `.reverse` al final |
| `_ + _` hace algo raro | cada `_` es un parámetro distinto | escribe la lambda completa |

Siguiente paso: **02-curry-howard.ipynb**.